# 0. Setting

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/multiplex-terror-network-gnn/

In [ ]:
! pip install torch-geometric --quiet

# 1. Data Generation

In [ ]:
# Baseline Dataset Generation
! python src/data/multiplex_generator_v2.py \
  --size 1500 \
  --seed 2025 \
  --out_dir data/multiplex_baseline \
  --config configs/generator_baseline.json

  # Baseline Data Transformation
! python src/data/build_pyg_dataset.py \
  --manifest data/multiplex_baseline/multiplex.json \
  --out_path data/multiplex_baseline/pyg_data.pt

In [ ]:
# Easy Dataset Generation
! python src/data/multiplex_generator_v2.py \
  --size 1500 \
  --seed 2025 \
  --out_dir data/multiplex_easy \
  --config configs/generator_baseline.json

  # Data Transformation
! python src/data/build_pyg_dataset.py \
  --manifest data/multiplex_easy/multiplex.json \
  --out_path data/multiplex_easy/pyg_data.pt

In [ ]:
# Hard Dataset Generation
! python src/data/multiplex_generator_v2.py \
  --size 1500 \
  --seed 2025 \
  --out_dir data/multiplex_hard \
  --config configs/generator_hard.json

  # Data Transformation
! python src/data/build_pyg_dataset.py \
  --manifest data/multiplex_hard/multiplex.json \
  --out_path data/multiplex_hard/pyg_data.pt

# 2. Data Statistics


In [ ]:
!python src/data/basic_diagnostics.py \
  --manifest data/multiplex_baseline/multiplex.json \
  --out_dir data/analysis/multiplex_baseline

# 3. GNN Models Solution

## 3-1. High-Value Target(HVT) Classification

In [ ]:
# HVT 단일 태스크
!python src/models/train_hvt_gnn.py \
  --data_path data/multiplex_baseline/pyg_data.pt \
  --hidden_dim 64 \
  --num_layers 2 \
  --dropout 0.5 \
  --pos_weight 5 \
  --lr 1e-3 \
  --weight_decay 1e-4 \
  --epochs 500 \
  --seed 2025

In [ ]:
# HVT 단일 태스크
!python src/models/train_hvt_gnn.py \
  --data_path data/multiplex_easy/pyg_data.pt \
  --hidden_dim 64 \
  --num_layers 2 \
  --dropout 0.5 \
  --pos_weight 5 \
  --lr 1e-3 \
  --weight_decay 1e-4 \
  --epochs 500 \
  --seed 2025

In [ ]:
# HVT 단일 태스크
!python src/models/train_hvt_gnn.py \
  --data_path data/multiplex_hard/pyg_data.pt \
  --hidden_dim 64 \
  --num_layers 2 \
  --dropout 0.5 \
  --pos_weight 5 \
  --lr 1e-3 \
  --weight_decay 1e-4 \
  --epochs 500 \
  --seed 2025

## 3-2. Multi-Tasking
 - Role Classification
 - HVT Classification
 - importance score Regression

In [ ]:
!python src/models/train_multitask_gnn.py \
  --data_path data/multiplex_baseline/pyg_data.pt \
  --hidden_dim 64 \
  --num_layers 3 \
  --dropout 0.5 \
  --lr 1e-3 \
  --weight_decay 5e-4 \
  --epochs 500 \
  --seed 2025 \
  --pos_weight 5.0 \
  --alpha_role 0.5 \
  --alpha_hvt 1.5 \
  --alpha_imp 0.1 \
  --patience 50 \
  --min_delta 1e-3

In [ ]:
!python src/models/train_multitask_gnn.py \
  --data_path data/multiplex_easy/pyg_data.pt \
  --hidden_dim 64 \
  --num_layers 3 \
  --dropout 0.5 \
  --lr 1e-3 \
  --weight_decay 5e-4 \
  --epochs 500 \
  --seed 2025 \
  --pos_weight 5.0 \
  --alpha_role 0.5 \
  --alpha_hvt 1.5 \
  --alpha_imp 0.1 \
  --patience 50 \
  --min_delta 1e-3

In [ ]:
!python src/models/train_multitask_gnn.py \
  --data_path data/multiplex_hard/pyg_data.pt \
  --hidden_dim 64 \
  --num_layers 3 \
  --dropout 0.5 \
  --lr 1e-3 \
  --weight_decay 5e-4 \
  --epochs 500 \
  --seed 2025 \
  --pos_weight 5.0 \
  --alpha_role 0.5 \
  --alpha_hvt 1.5 \
  --alpha_imp 0.1 \
  --patience 50 \
  --min_delta 1e-3

## 3-3. Specific node link prediction
 - Finance layer illegal fund link prediction
 - Communication Layer Contact Prediction


In [ ]:
# Finance Layer Uniform vs Hard Region (불법자금 링크 예측)
!python src/models/train_linkpred_layer.py \
  --data_path data/multiplex_baseline/pyg_data.pt \
  --layer finance \
  --hidden_dim 64 \
  --num_layers 2 \
  --dropout 0.3 \
  --lr 1e-3 \
  --weight_decay 1e-4 \
  --epochs 500 \
  --seed 2025 \
  --patience 50 \
  --min_delta 1e-3 \
  --neg_mode uniform

# 2. hard_region negative
!python src/models/train_linkpred_layer.py \
  --data_path data/multiplex_baseline/pyg_data.pt \
  --layer finance \
  --hidden_dim 64 \
  --num_layers 2 \
  --dropout 0.3 \
  --lr 1e-3 \
  --weight_decay 1e-4 \
  --epochs 500 \
  --seed 2025 \
  --patience 50 \
  --min_delta 1e-3 \
  --neg_mode hard_region

In [ ]:
# Finance Layer Uniform vs Hard Region
!python src/models/train_linkpred_layer.py \
  --data_path data/multiplex_easy/pyg_data.pt \
  --layer finance \
  --hidden_dim 64 \
  --num_layers 2 \
  --dropout 0.3 \
  --lr 1e-3 \
  --weight_decay 1e-4 \
  --epochs 500 \
  --seed 2025 \
  --patience 50 \
  --min_delta 1e-3 \
  --neg_mode uniform

# 2. hard_region negative
!python src/models/train_linkpred_layer.py \
  --data_path data/multiplex_easy/pyg_data.pt \
  --layer finance \
  --hidden_dim 64 \
  --num_layers 2 \
  --dropout 0.3 \
  --lr 1e-3 \
  --weight_decay 1e-4 \
  --epochs 500 \
  --seed 2025 \
  --patience 50 \
  --min_delta 1e-3 \
  --neg_mode hard_region

In [ ]:
# Finance Layer Uniform vs Hard Region
!python src/models/train_linkpred_layer.py \
  --data_path data/multiplex_hard/pyg_data.pt \
  --layer finance \
  --hidden_dim 64 \
  --num_layers 2 \
  --dropout 0.3 \
  --lr 1e-3 \
  --weight_decay 1e-4 \
  --epochs 500 \
  --seed 2025 \
  --patience 50 \
  --min_delta 1e-3 \
  --neg_mode uniform

# 2. hard_region negative
!python src/models/train_linkpred_layer.py \
  --data_path data/multiplex_hard/pyg_data.pt \
  --layer finance \
  --hidden_dim 64 \
  --num_layers 2 \
  --dropout 0.3 \
  --lr 1e-3 \
  --weight_decay 1e-4 \
  --epochs 500 \
  --seed 2025 \
  --patience 50 \
  --min_delta 1e-3 \
  --neg_mode hard_region

In [ ]:
# Communication Layer Uniform
!python src/models/train_linkpred_layer.py \
  --data_path data/multiplex_baseline/pyg_data.pt \
  --layer communication \
  --hidden_dim 64 \
  --num_layers 2 \
  --dropout 0.3 \
  --lr 1e-3 \
  --weight_decay 1e-4 \
  --epochs 500 \
  --seed 2025 \
  --patience 50 \
  --min_delta 1e-3 \
  --neg_mode uniform

# hard_region
!python src/models/train_linkpred_layer.py \
  --data_path data/multiplex_baseline/pyg_data.pt \
  --layer communication \
  --hidden_dim 64 \
  --num_layers 2 \
  --dropout 0.3 \
  --lr 1e-3 \
  --weight_decay 1e-4 \
  --epochs 500 \
  --seed 2025 \
  --patience 50 \
  --min_delta 1e-3 \
  --neg_mode hard_region

In [ ]:
# Communication Layer Uniform
!python src/models/train_linkpred_layer.py \
  --data_path data/multiplex_easy/pyg_data.pt \
  --layer communication \
  --hidden_dim 64 \
  --num_layers 2 \
  --dropout 0.3 \
  --lr 1e-3 \
  --weight_decay 1e-4 \
  --epochs 500 \
  --seed 2025 \
  --patience 50 \
  --min_delta 1e-3 \
  --neg_mode uniform

# hard_region
!python src/models/train_linkpred_layer.py \
  --data_path data/multiplex_easy/pyg_data.pt \
  --layer communication \
  --hidden_dim 64 \
  --num_layers 2 \
  --dropout 0.3 \
  --lr 1e-3 \
  --weight_decay 1e-4 \
  --epochs 500 \
  --seed 2025 \
  --patience 50 \
  --min_delta 1e-3 \
  --neg_mode hard_region

In [ ]:
# Communication Layer Uniform
!python src/models/train_linkpred_layer.py \
  --data_path data/multiplex_hard/pyg_data.pt \
  --layer communication \
  --hidden_dim 64 \
  --num_layers 2 \
  --dropout 0.3 \
  --lr 1e-3 \
  --weight_decay 1e-4 \
  --epochs 500 \
  --seed 2025 \
  --patience 50 \
  --min_delta 1e-3 \
  --neg_mode uniform

# hard_region
!python src/models/train_linkpred_layer.py \
  --data_path data/multiplex_hard/pyg_data.pt \
  --layer communication \
  --hidden_dim 64 \
  --num_layers 2 \
  --dropout 0.3 \
  --lr 1e-3 \
  --weight_decay 1e-4 \
  --epochs 500 \
  --seed 2025 \
  --patience 50 \
  --min_delta 1e-3 \
  --neg_mode hard_region

# 4. Result Visualization

In [ ]:
# Plot
!python src/analysis/plot_multitask_linkpred_summary.py \
  --run_dirs \
    data/multiplex_easy \
    data/multiplex_baseline \
    data/multiplex_hard \
  --out_dir results/summary_all